El objetivo de este código es medir la tensión para $H_0$ utilizando distintos datsasets, mediante la métrica $Q_{DM}$

In [ ]:
#Librerias:

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from getdist import loadMCSamples
from scipy.stats import chi2
from scipy.special import erfcinv


In [ ]:
#Buscamos cargar los datsets:

def cargar_supernovas(ruta_archivo, col_omega, col_h0, multiplicar_h0=False):
    """
    Carga los datos de supernovas desde un archivo CSV.

    Parámetros:
    ruta_archivo (str): Ruta al archivo CSV que contiene los datos de supernovas.
    col_omega (str): Nombre de la columna que contiene los valores de Omega_m.
    col_h0 (str): Nombre de la columna que contiene los valores de H0.
    multiplicar_h0 (bool): Si es True, se multiplicarán los valores de H0 por 100.
    """
    # Cargar el dataset 
    df = pd.read_csv(ruta_archivo, comment='#', sep='\s+') #los archivos Phantheos tienen un # para los comentarios
    

    # Definimos las columnas:
    omega_m_data = df[col_omega]
    h0_data = df[col_h0]
    # Multiplicar H0 por 100 si se indica
    if multiplicar_h0:
        h0_data *= 100
    
    #Buscamos las medias y covarianzas de omega y H_0:
    datos = np.vstack([omega_m_data, h0_data])
    media = np.mean(datos, axis=1)
    covarianza = np.cov(datos)

    return media, covarianza

In [ ]:
# ==========================================================
# Celda 2: Llamando a los archivos (Planck y Pantheon)
# ==========================================================

print("1. Procesando archivo de Supernovas...")
# Aquí usamos TU función. Fíjate que le pasamos el nombre exacto de las columnas de Pantheon
ruta_sn = "C:\\Users\\hijos\\Investigacion 1\\Pantheon+SH0ES_LambdaCDM.txt"
media_sn, cov_sn = cargar_supernovas(ruta_sn, col_omega='omega_m', col_h0='h0', multiplicar_h0=True)

print(f"   -> Centro SN: Omega_m = {media_sn[0]:.3f}, H0 = {media_sn[1]:.2f}")


print("\n2. Procesando cadenas de Planck...")
# Recuerda que para Planck, getdist lee la "raíz" del archivo (sin el _1.txt)
ruta_planck = "Planck/base/plikHM_TTTEEE_lowl_lensing/base_plikHM_TTTEEE_lowl_lensing"

# Usamos la librería loadMCSamples para leer todas las cadenas de Planck de golpe
cadenas_planck = loadMCSamples(ruta_planck, settings={'ignore_rows': 0.3})
parametros_p = cadenas_planck.getParams()

# Extraemos la media y covarianza, recordando que Planck usa "pesos"
datos_planck = np.vstack([parametros_p.omegam, parametros_p.H0])
media_planck = np.average(datos_planck, weights=cadenas_planck.weights, axis=1)
cov_planck = np.cov(datos_planck, aweights=cadenas_planck.weights)

print(f"   -> Centro Planck: Omega_m = {media_planck[0]:.3f}, H0 = {media_planck[1]:.2f}")